In [1]:
# MiniGPT — Fine-Tuning

# This notebook performs a controlled fine-tuning phase on the pre-trained character-level MiniGPT model.

# Goals:
#1. Load the trained baseline model.
#2. Load the baseline evaluation results.
#3. Continue training with a smaller learning rate.
#4. Monitor training and validation loss.
#5. Save the fine-tuned model.
#6. Compare fine-tuned performance with the baseline.

In [1]:
import os
import json
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.14.0


NameError: name 'device' is not defined

In [2]:
device = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: mps


In [3]:
# Paths

CHECKPOINT_PATH = "../models/minigpt_checkpoint.pt"
BASELINE_PATH = "../models/baseline_evaluation.json"

print("Checkpoint:", CHECKPOINT_PATH)
print("Baseline:", BASELINE_PATH)

print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))
print("Baseline exists:", os.path.exists(BASELINE_PATH))

Checkpoint: ../models/minigpt_checkpoint.pt
Baseline: ../models/baseline_evaluation.json
Checkpoint exists: True
Baseline exists: True


In [4]:
with open(BASELINE_PATH, "r") as file:
    baseline_results = json.load(file)

print("Baseline evaluation loaded:")
print(json.dumps(baseline_results, indent=4))

Baseline evaluation loaded:
{
    "parameters": 3221567,
    "training_loss": 1.3509242272377013,
    "validation_loss": 1.5692376387119293,
    "training_perplexity": 3.8609923164462248,
    "validation_perplexity": 4.80298518771129,
    "training_accuracy": 0.581708984375,
    "validation_accuracy": 0.5351708984375
}


In [5]:
# Model configuration

vocab_size = 63
context_length = 128

embedding_dim = 256
num_heads = 4
num_layers = 4

dropout = 0.2

print("Model Configuration")
print("=" * 40)
print("Vocabulary size     :", vocab_size)
print("Context length      :", context_length)
print("Embedding dimension :", embedding_dim)
print("Attention heads     :", num_heads)
print("Transformer layers  :", num_layers)
print("Dropout             :", dropout)

Model Configuration
Vocabulary size     : 63
Context length      : 128
Embedding dimension : 256
Attention heads     : 4
Transformer layers  : 4
Dropout             : 0.2


In [6]:
class Head(nn.Module):

    def __init__(self, embedding_dim, head_dim, context_length, dropout):
        super().__init__()

        self.key = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.query = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.value = nn.Linear(
            embedding_dim,
            head_dim,
            bias=False
        )

        self.dropout = nn.Dropout(dropout)

        self.register_buffer(
            "tril",
            torch.tril(
                torch.ones(
                    context_length,
                    context_length
                )
            )
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        attention_scores = q @ k.transpose(-2, -1)

        attention_scores = (
            attention_scores
            * (k.shape[-1] ** -0.5)
        )

        attention_scores = attention_scores.masked_fill(
            self.tril[:T, :T] == 0,
            float("-inf")
        )

        attention_weights = F.softmax(
            attention_scores,
            dim=-1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        v = self.value(x)

        output = attention_weights @ v

        return output

In [7]:
class MultiHeadAttention(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        head_dim = embedding_dim // num_heads

        self.heads = nn.ModuleList([
            Head(
                embedding_dim,
                head_dim,
                context_length,
                dropout
            )
            for _ in range(num_heads)
        ])

        self.projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        outputs = [
            head(x)
            for head in self.heads
        ]

        x = torch.cat(outputs, dim=-1)

        x = self.projection(x)

        x = self.dropout(x)

        return x

In [8]:
class FeedForward(nn.Module):

    def __init__(
        self,
        embedding_dim,
        dropout
    ):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(
                embedding_dim,
                4 * embedding_dim
            ),

            nn.GELU(),

            nn.Linear(
                4 * embedding_dim,
                embedding_dim
            ),

            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.network(x)

In [9]:
class TransformerBlock(nn.Module):

    def __init__(
        self,
        embedding_dim,
        num_heads,
        context_length,
        dropout
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(
            embedding_dim
        )

        self.attention = MultiHeadAttention(
            embedding_dim,
            num_heads,
            context_length,
            dropout
        )

        self.ln2 = nn.LayerNorm(
            embedding_dim
        )

        self.ffn = FeedForward(
            embedding_dim,
            dropout
        )

    def forward(self, x):

        x = x + self.attention(
            self.ln1(x)
        )

        x = x + self.ffn(
            self.ln2(x)
        )

        return x

In [10]:
class MiniGPT(nn.Module):

    def __init__(
        self,
        vocab_size,
        context_length,
        embedding_dim,
        num_heads,
        num_layers,
        dropout
    ):
        super().__init__()

        self.context_length = context_length

        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim
        )

        self.position_embedding = nn.Embedding(
            context_length,
            embedding_dim
        )

        self.blocks = nn.Sequential(
            *[
                TransformerBlock(
                    embedding_dim,
                    num_heads,
                    context_length,
                    dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.ln_final = nn.LayerNorm(
            embedding_dim
        )

        self.lm_head = nn.Linear(
            embedding_dim,
            vocab_size
        )

    def forward(self, idx):

        B, T = idx.shape

        token_embeddings = self.token_embedding(idx)

        positions = torch.arange(
            T,
            device=idx.device
        )

        position_embeddings = self.position_embedding(
            positions
        )

        x = token_embeddings + position_embeddings

        x = self.blocks(x)

        x = self.ln_final(x)

        logits = self.lm_head(x)

        return logits

In [11]:
model = MiniGPT(
    vocab_size=vocab_size,
    context_length=context_length,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    dropout=dropout
)

model = model.to(device)

print("Model created.")

Model created.


In [12]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print("Total parameters:", f"{total_parameters:,}")

Total parameters: 3,221,567


In [13]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:")
    print(checkpoint.keys())

Checkpoint type: <class 'dict'>
Checkpoint keys:
dict_keys(['iteration', 'model_state_dict', 'optimizer_state_dict', 'train_loss', 'val_loss', 'vocab', 'stoi', 'itos'])


In [14]:
model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)

print("Trained model weights loaded successfully.")
print("Training iteration:", checkpoint["iteration"])
print("Training loss:", checkpoint["train_loss"])
print("Validation loss:", checkpoint["val_loss"])

Trained model weights loaded successfully.
Training iteration: 5000
Training loss: 1.3438644409179688
Validation loss: 1.4844762086868286


In [15]:
# Fine-tuning optimizer

learning_rate = 1e-4

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=0.01
)

print("Fine-tuning optimizer created.")
print("Learning rate:", learning_rate)

Fine-tuning optimizer created.
Learning rate: 0.0001


In [16]:
optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

print("Optimizer state restored.")

Optimizer state restored.


In [18]:
def get_batch(data, batch_size=32, context_length=128):
    starts = torch.randint(
        0,
        len(data) - context_length - 1,
        (batch_size,)
    )

    x = torch.stack([
        data[start:start + context_length]
        for start in starts
    ])

    y = torch.stack([
        data[start + 1:start + context_length + 1]
        for start in starts
    ])

    return x, y

print("get_batch() restored successfully.")

get_batch() restored successfully.


In [21]:
import os

print("Files in ../data:")
print(os.listdir("../data"))

Files in ../data:
['processed', 'raw']


In [23]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles/folders here:")
for item in os.listdir("."):
    print(item)

Current directory:
/Users/anjana/Desktop/Industry Projects/LLM Train/Project 1/notebooks

Files/folders here:
06_train_model.ipynb
05_build_transformer.ipynb
08_evaluation.ipynb
02_data_cleaning.ipynb
07_generate_text.ipynb
09_fine_tuning.ipynb
.ipynb_checkpoints
01_dataset_exploration.ipynb
03_tokenization.ipynb
04_dataset_preparation.ipynb


In [25]:
import os

project_root = ".."

print("Project 1 contents:")
for item in sorted(os.listdir(project_root)):
    print(item)

Project 1 contents:
.ipynb_checkpoints
.venv
3.22-million-parameter decoder.md
LLM_Structure.md
data
learning_path.md
models
notebooks
project_structure.md
venv


In [26]:
import os

DATA_DIR = "../data"

print("Data folder contents:")
for item in sorted(os.listdir(DATA_DIR)):
    print(item)

Data folder contents:
processed
raw


In [27]:
MODEL_DIR = "../models"

print("Models folder contents:")
for item in sorted(os.listdir(MODEL_DIR)):
    print(item)

Models folder contents:
baseline_evaluation.json
minigpt_checkpoint.pt


In [28]:
import os

PROCESSED_DIR = "../data/processed"

print("Processed data:")
for item in sorted(os.listdir(PROCESSED_DIR)):
    print(item)

Processed data:
tiny-shakespeare_clean.txt


In [29]:
# ============================================================
# RESTORE CLEANED CORPUS
# ============================================================

CLEAN_TEXT_PATH = "../data/processed/tiny-shakespeare_clean.txt"

with open(CLEAN_TEXT_PATH, "r", encoding="utf-8") as f:
    text = f.read()

print("Cleaned corpus loaded.")
print("Characters:", len(text))
print("First 200 characters:")
print(text[:200])

Cleaned corpus loaded.
Characters: 1114961
First 200 characters:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [30]:
# ============================================================
# RESTORE TOKENIZER + DATASET
# ============================================================

# Restore the exact vocabulary used during training
stoi = checkpoint["stoi"]
itos = checkpoint["itos"]
vocab = checkpoint["vocab"]

print("Vocabulary size:", len(vocab))
print("Vocabulary restored from checkpoint.")

# Encode cleaned text using the original tokenizer
encoded_data = torch.tensor(
    [stoi[ch] for ch in text],
    dtype=torch.long
)

print("Encoded data created.")
print("Total tokens:", len(encoded_data))

# Same train/validation split used for the original training
split_idx = int(0.9 * len(encoded_data))

train_data = encoded_data[:split_idx]
val_data = encoded_data[split_idx:]

print()
print("Dataset split:")
print("Training tokens  :", len(train_data))
print("Validation tokens:", len(val_data))

Vocabulary size: 63
Vocabulary restored from checkpoint.
Encoded data created.
Total tokens: 1114961

Dataset split:
Training tokens  : 1003464
Validation tokens: 111497


In [32]:
# ============================================================
# RESTORE BATCH CONFIGURATION + BATCH GENERATOR
# ============================================================

batch_size = 32
context_length = 128


def get_batch(data, batch_size=32, context_length=128):

    starts = torch.randint(
        0,
        len(data) - context_length - 1,
        (batch_size,)
    )

    x = torch.stack([
        data[start:start + context_length]
        for start in starts
    ])

    y = torch.stack([
        data[start + 1:start + context_length + 1]
        for start in starts
    ])

    return x, y


# Test
x_test, y_test = get_batch(
    train_data,
    batch_size=batch_size,
    context_length=context_length
)

print("Batch generator restored successfully.")
print("Batch size:", batch_size)
print("Context length:", context_length)
print("X shape:", x_test.shape)
print("Y shape:", y_test.shape)

Batch generator restored successfully.
Batch size: 32
Context length: 128
X shape: torch.Size([32, 128])
Y shape: torch.Size([32, 128])


In [33]:
# ============================================================
# FINE-TUNING LOOP
# ============================================================

fine_tune_steps = 1000

model.train()

fine_tune_losses = []
fine_tune_val_losses = []

print("Starting fine-tuning...")
print("=" * 60)

start_time = time.time()

for step in range(fine_tune_steps):

    # Get training batch
    x_batch, y_batch = get_batch(
        train_data,
        batch_size=batch_size,
        context_length=context_length
    )

    x_batch = x_batch.to(device)
    y_batch = y_batch.to(device)

    # Clear gradients
    optimizer.zero_grad()

    # Forward pass
    logits = model(x_batch)

    # Calculate loss
    loss = F.cross_entropy(
        logits.view(-1, vocab_size),
        y_batch.view(-1)
    )

    # Backward pass
    loss.backward()

    # Update parameters
    optimizer.step()

    # Record training loss
    fine_tune_losses.append(loss.item())

    # Validation every 100 steps
    if step % 100 == 0:

        model.eval()

        with torch.no_grad():

            x_val, y_val = get_batch(
                val_data,
                batch_size=batch_size,
                context_length=context_length
            )

            x_val = x_val.to(device)
            y_val = y_val.to(device)

            val_logits = model(x_val)

            val_loss = F.cross_entropy(
                val_logits.view(-1, vocab_size),
                y_val.view(-1)
            )

            fine_tune_val_losses.append(val_loss.item())

        model.train()

        elapsed = time.time() - start_time

        print(
            f"Step {step:4d} | "
            f"Train Loss: {loss.item():.4f} | "
            f"Val Loss: {val_loss.item():.4f} | "
            f"Time: {elapsed:.1f}s"
        )

print("=" * 60)
print("Fine-tuning complete.")

Starting fine-tuning...
Step    0 | Train Loss: 1.4132 | Val Loss: 1.5954 | Time: 4.0s
Step  100 | Train Loss: 1.4523 | Val Loss: 1.5829 | Time: 23.9s
Step  200 | Train Loss: 1.4981 | Val Loss: 1.5931 | Time: 44.0s
Step  300 | Train Loss: 1.4522 | Val Loss: 1.5566 | Time: 64.1s
Step  400 | Train Loss: 1.4310 | Val Loss: 1.4850 | Time: 84.0s
Step  500 | Train Loss: 1.4466 | Val Loss: 1.6006 | Time: 104.1s
Step  600 | Train Loss: 1.4205 | Val Loss: 1.5513 | Time: 124.1s
Step  700 | Train Loss: 1.4594 | Val Loss: 1.5933 | Time: 144.4s
Step  800 | Train Loss: 1.3871 | Val Loss: 1.5580 | Time: 164.1s
Step  900 | Train Loss: 1.3874 | Val Loss: 1.5663 | Time: 184.2s
Fine-tuning complete.


In [34]:
# ============================================================
# SAVE FINE-TUNED MODEL
# ============================================================

FINE_TUNED_PATH = "../models/minigpt_finetuned_checkpoint.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "iteration": fine_tune_steps,
        "train_loss": fine_tune_losses[-1],
        "val_loss": fine_tune_val_losses[-1],
        "vocab": checkpoint["vocab"],
        "stoi": checkpoint["stoi"],
        "itos": checkpoint["itos"],
    },
    FINE_TUNED_PATH
)

print("Fine-tuned model saved successfully.")
print("Path:", FINE_TUNED_PATH)
print("Exists:", os.path.exists(FINE_TUNED_PATH))

Fine-tuned model saved successfully.
Path: ../models/minigpt_finetuned_checkpoint.pt
Exists: True


In [35]:
# ============================================================
# EVALUATE FINE-TUNED MODEL
# ============================================================

model.eval()

eval_steps = 100

fine_tune_eval_losses = []

with torch.no_grad():

    for step in range(eval_steps):

        x_batch, y_batch = get_batch(
            val_data,
            batch_size=batch_size,
            context_length=context_length
        )

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(x_batch)

        loss = F.cross_entropy(
            logits.view(-1, vocab_size),
            y_batch.view(-1)
        )

        fine_tune_eval_losses.append(loss.item())


fine_tuned_val_loss = sum(fine_tune_eval_losses) / len(fine_tune_eval_losses)

fine_tuned_perplexity = torch.exp(
    torch.tensor(fine_tuned_val_loss)
).item()


print("=" * 60)
print("FINE-TUNED MODEL EVALUATION")
print("=" * 60)

print(f"Validation Loss       : {fine_tuned_val_loss:.4f}")
print(f"Validation Perplexity : {fine_tuned_perplexity:.2f}")

FINE-TUNED MODEL EVALUATION
Validation Loss       : 1.5435
Validation Perplexity : 4.68


In [36]:
# ============================================================
# FINE-TUNED MODEL ACCURACY
# ============================================================

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for step in range(100):

        x_batch, y_batch = get_batch(
            val_data,
            batch_size=batch_size,
            context_length=context_length
        )

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(x_batch)

        predictions = torch.argmax(logits, dim=-1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.numel()

fine_tuned_accuracy = correct / total

print("=" * 60)
print("FINE-TUNED MODEL ACCURACY")
print("=" * 60)
print(f"Validation Accuracy : {fine_tuned_accuracy * 100:.2f}%")

FINE-TUNED MODEL ACCURACY
Validation Accuracy : 54.46%


In [37]:
# ============================================================
# LOAD BASELINE MODEL FOR COMPARISON
# ============================================================

baseline_model = MiniGPT(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    context_length=context_length,
    dropout=dropout
).to(device)

baseline_checkpoint = torch.load(
    "../models/minigpt_checkpoint.pt",
    map_location=device,
    weights_only=False
)

baseline_model.load_state_dict(
    baseline_checkpoint["model_state_dict"]
)

baseline_model.eval()

print("Baseline model loaded.")

Baseline model loaded.


In [38]:
# ============================================================
# GENERATION FUNCTION
# ============================================================

def generate_from_model(
    model,
    prompt,
    max_new_tokens=200,
    temperature=0.8,
    top_k=20
):
    model.eval()

    # Encode prompt
    idx = torch.tensor(
        [encode(prompt)],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):

        # Keep only context window
        idx_cond = idx[:, -context_length:]

        with torch.no_grad():
            logits = model(idx_cond)

        # Get last-token logits
        logits = logits[:, -1, :]

        # Temperature
        logits = logits / temperature

        # Top-k sampling
        if top_k is not None:
            values, indices = torch.topk(
                logits,
                min(top_k, logits.size(-1))
            )

            filtered_logits = torch.full_like(
                logits,
                float("-inf")
            )

            filtered_logits.scatter_(
                1,
                indices,
                values
            )

            logits = filtered_logits

        # Convert to probabilities
        probabilities = torch.softmax(logits, dim=-1)

        # Sample next token
        next_token = torch.multinomial(
            probabilities,
            num_samples=1
        )

        idx = torch.cat(
            [idx, next_token],
            dim=1
        )

    return decode(idx[0].tolist())

In [40]:
# ============================================================
# RESTORE TOKENIZER
# ============================================================

stoi = checkpoint["stoi"]
itos = checkpoint["itos"]

def encode(text):
    return [stoi[ch] for ch in text]

def decode(tokens):
    return "".join(itos[token] for token in tokens)

print("Tokenizer restored.")
print("Vocabulary size:", len(stoi))

Tokenizer restored.
Vocabulary size: 63


In [41]:
# ============================================================
# BASELINE vs FINE-TUNED
# ============================================================

prompts = [
    "ROMEO:",
    "HAMLET:",
    "KING:",
    "LORD:",
    "What",
    "To be"
]

for prompt in prompts:

    print("=" * 70)
    print("PROMPT:", prompt)
    print("=" * 70)

    print("\n--- BASELINE MODEL ---\n")

    baseline_output = generate_from_model(
        baseline_model,
        prompt,
        max_new_tokens=150,
        temperature=0.8,
        top_k=20
    )

    print(baseline_output)

    print("\n--- FINE-TUNED MODEL ---\n")

    finetuned_output = generate_from_model(
        model,
        prompt,
        max_new_tokens=150,
        temperature=0.8,
        top_k=20
    )

    print(finetuned_output)
    print()

PROMPT: ROMEO:

--- BASELINE MODEL ---

ROMEO:
Is it were to see your worship.

ANGELO:
For him, what he dost present be thy foot.

Clown:
Let me not should have been with clouds
Which the armed o

--- FINE-TUNED MODEL ---

ROMEO:
Why, she is my word, being these did his husbands
Betwixt him not that have heard him he heads.

HENRY BOLINGBROKE:
No more captive you might of the 

PROMPT: HAMLET:

--- BASELINE MODEL ---

HAMLET:
No should not all great my strike of the bast
Hall been well were a must the entreat forget,
Even and the cap ancient worm of my kind
And into bless 

--- FINE-TUNED MODEL ---

HAMLET:
Shall I will be a follow to him: he would be many be
In my father black'd with you.

KING EDWARD IV:
O, then, as let you long my death your bruite to

PROMPT: KING:

--- BASELINE MODEL ---

KING:
By my marker to my command;
If of your rospition and never and leave tie,
That are your offends stood extremy robs.
If this a grown honour works, and

--- FINE-TUNED MODEL ---

KING:
Your 

In [42]:
for prompt in prompts:

    print("=" * 70)
    print("PROMPT:", prompt)
    print("=" * 70)

    print("\n--- BASELINE MODEL ---\n")

    baseline_output = generate_from_model(
        baseline_model,
        prompt,
        max_new_tokens=150,
        temperature=0.8,
        top_k=20
    )

    print(baseline_output)

    print("\n--- FINE-TUNED MODEL ---\n")

    finetuned_output = generate_from_model(
        model,
        prompt,
        max_new_tokens=150,
        temperature=0.8,
        top_k=20
    )

    print(finetuned_output)
    print()

PROMPT: ROMEO:

--- BASELINE MODEL ---

ROMEO:
Why, you are the good good, sir, persuaring by the faults
She did with dishonour'd with thy again.

AUTOLYCUS:
They shall heer these will the raise c

--- FINE-TUNED MODEL ---

ROMEO:
He doubt sure me, that I am too much affair it
To strong and remain the winterent of his house,
And what a grace off soldier and repent
Have cannot b

PROMPT: HAMLET:

--- BASELINE MODEL ---

HAMLET:
Be the sun that may me to stold again:
And I think thee, one to much as what they are are here?

CORIOLANUS:
Is the speak at heaven had with a duke h

--- FINE-TUNED MODEL ---

HAMLET:
Ah, he should have not would not my suitor.

CORIOLANUS:
Come, I had longer you.

PAULINA:
I do not forth,
But I have rather not for the noble hearts

PROMPT: KING:

--- BASELINE MODEL ---

KING:
And if his purpose for a battle law,
The good drone of his humblingly of this chamber,
Or speak in his suitors and long himself
To that thou didst yo

--- FINE-TUNED MODEL ---

KING:
Yea, 